In [ ]:
///- --package . --timeout 900000

#### gpu_sim (test)

In [ ]:
///- --test static --symbol

:< testing

In [ ]:
:= troll_camp () : hunting_pack::camp = hunting_pack::uniform_camp (:< creatures ;; Troll)

:= reach_first () : hunting_pack::tactic = hunting_pack::cavebot (:< policy ;; ReachFirst)

:~ ^ troll_ticks (seed : u32) (tick : i32) (p : hunting_pack::pack) : hunting_pack::pack =
    ? tick >= 10
    ?> p
    |: troll_ticks seed (tick + 1) (hunting_pack::advance (troll_camp ()) (reach_first ()) seed tick p)

:= ten_tick_facts (seed : u32) : i32 * i32 * i32 * i32 =
    := p = troll_ticks seed 0 (hunting_pack::start (troll_camp ()))
    units::value_of p::hunter::health,
    (hunting_pack::slot 0 p)::at::x,
    (hunting_pack::slot 0 p)::at::y,
    p::score::kills

#### ten_ticks_of_a_troll_pack_on_the_oracle (test)

In [1]:
///- --test --symbol
///> c

:= ~seed = 7u32
ten_tick_facts seed |> _assert_eq (185i32, 105i32, 102i32, 0i32)

__assert_eq { expected = (185, 105, 102, 0) }

#### ten_ticks_of_a_troll_pack_on_gpu (test)

In [2]:
///- --test --symbol
///> wgsl

:= ~seed = 7u32
:= health, x, y, kills = ten_tick_facts seed
? health = 185i32 && x = 105i32 && y = 102i32 && kills = 0i32 ?> 0i32 |: 1i32

In [ ]:
:= spot_clears (seed : u32) =
    :< hunting_pack
    := spot = uniform_camp (:< creatures ;; Cyclops) |> on_layout (:< terrain ;; ArenaSpot)
    clear_ledger_of spot (cavebot_with_heal 160) (seed + 994u32) 5

:= troll_hunt (seed : u32) : hunting_pack::pack =
    hunting_pack::simulate (troll_camp ()) (reach_first ()) seed (hunting_pack::scenario_ticks ())

:= hunt_and_clear_facts (seed : u32) : i32 * i32 * i32 * i32 * i32 * i32 =
    := hunt, c = troll_hunt seed, spot_clears seed
    hunt::score::kills,
    hunt::score::escapes,
    hunt::score::deaths,
    c::clear_potions,
    c::clear_deaths,
    c::clear_ms

#### troll_hunt_and_cyclops_clears_on_the_oracle (test)

In [3]:
///- --test --symbol
///> c

:= ~seed = 7u32
hunt_and_clear_facts seed |> _assert_eq (6i32, 18i32, 0i32, 84i32, 0i32, 453000i32)

__assert_eq { expected = (6, 18, 0, 84, 0, 453000) }

#### troll_hunt_and_cyclops_clears_on_gpu (test)

In [4]:
///- --test --symbol
///> wgsl

:= ~seed = 7u32
:= kills, fled, deaths, potions, lost, ms = hunt_and_clear_facts seed
? kills = 6i32 && fled = 18i32 && deaths = 0i32 && potions = 84i32 && lost = 0i32 && ms = 453000i32 ?> 0i32 |: 1i32

In [ ]:
:= routed_hunt (seed : u32) =
    :< hunting_pack
    simulate (arena_hunt ()) (cavebot_with_heal 110 |> sweeping_spawns) seed (scenario_ticks ())

#### a_hunting_ground_hunt_with_routing_on_the_oracle (test)

In [5]:
///- --test --symbol
///> c

:= ~seed = 1001u32
:= p = routed_hunt seed
(p::score::kills, p::score::experience, p::hunter::tally::potions, p::score::deaths)
|> _assert_eq (23i32, 1370i32, 24i32, 0i32)

__assert_eq { expected = (23, 1370, 24, 0) }

In [ ]:
:= routed_hunt_matches (seed : u32) : bool =
    := p = routed_hunt seed
    p::score::kills = 23i32
    && p::score::experience = 1370i32
    && p::hunter::tally::potions = 24i32
    && p::score::deaths = 0i32

#### a_hunting_ground_hunt_with_routing_on_gpu (test)

In [6]:
///- --test --symbol
///> wgsl

:= ~seed = 1001u32
? routed_hunt_matches seed ?> 0i32 |: 1i32